In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, HMCGibbs

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp import(
    kalman_filter_1d,
)

In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
regressors

In [ ]:
response = df["sales"].values
response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)

X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)
Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="Log Sales")

In [ ]:
def make_fourier_series(t, period, order):
    """
    Args
    ----
    t: array-like, time points at which to evaluate the Fourier series
    period: int, the period of the seasonality
    order: int, the number of Fourier terms to include
    """
    sin_terms = jnp.array([jnp.sin(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    cos_terms = jnp.array([jnp.cos(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    return jnp.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

x_glb_trend = jnp.arange(len(y)) / 365.25
x_annual_seas = make_fourier_series(jnp.arange(len(y)), period=365.25, order=3)
x_weekly_seas = make_fourier_series(jnp.arange(len(y)), period=7, order=2)
x_seas = jnp.concatenate((x_annual_seas, x_weekly_seas), axis=1)
print(x_seas.shape)

In [ ]:
a0 = jnp.array([0.0] + [0.1] * (X.shape[-1]))
P0 = jnp.array([1.0] + [0.1] * (X.shape[-1]))

sigma_q_loc_prior = np.array([0.05] + [0.1] * (X.shape[-1]))
sigma_q_scale_prior = np.array([0.01] + [0.05] * (X.shape[-1]))

print("a0 shape:", a0.shape)
print("P0 shape:", P0.shape)

In [ ]:
# Test run to verify dimensions are correct before sampling
sigma_h = jnp.array(0.01)
sigma_q = jnp.full(Z.shape[-1], 0.05)

lp, at, Pt, _, _, _ = kalman_filter_1d(
    a0=a0,
    P0=P0,
    sigma_h=sigma_h,
    sigma_q=sigma_q,
    y=y,
    Z=Z,
    logp=True,
)
print("lp:", lp)
print("at shape:", at.shape)
print("Pt shape:", Pt.shape)

In [ ]:
def _nuts_fn(a0, P0):
    n_steps = len(y)
    n_states = P0.shape[0]
    # steps_plate = numpyro.plate("steps", n_steps, dim=-1)
    sigma_h = numpyro.sample(
        'sigma_q',
        dist.TruncatedNormal(
            0.01, 1.0, high=1.0, low=1e-5
        )
    )

    sigma_q = numpyro.sample(
        'sigma_h',
        dist.TruncatedNormal(
            sigma_q_loc_prior, sigma_q_scale_prior, high=1.0, low=1e-5
        )
    )

    lp, at, _, _, _, _ = kalman_filter_1d(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        # y=y - reg_comp,
        y=y,
        Z=Z,
        logp=True,
    )

    # placeholder for gibbs
    # numpyro.sample("smooth_alpha", dist.Normal(), sample_shape=(n_steps, ))

    numpyro.factor("lp", lp)
    numpyro.deterministic("at", at)
    numpyro.deterministic("mu", jnp.sum(Z * at, -1))

# def _gibbs_fn(rng_key, gibbs_sites, hmc_sites):
#     a0 = hmc_sites["a0"]
#     P0 = hmc_sites["P0"]
#     sigma_h = hmc_sites["sigma_h"]
#     sigma_q = hmc_sites["sigma_q"]

#     smooth_alpha = kalman_smooth(
#         a0=a0,
#         P0=P0,
#         sigma_h=sigma_h,
#         sigma_q=sigma_q,
#         obs=y,
#         rng_key=rng_key,
#     )
    
#     return {
#         "smooth_alpha": smooth_alpha,
#     }

Not sure why new jax and numpyro takes so long to run

In [ ]:
nuts_kernel = NUTS(_nuts_fn)
# kernel = HMCGibbs(
#     inner_kernel=nuts_kernel,
#     gibbs_fn=_gibbs_fn,
#     gibbs_sites=["smooth_alpha"],
# )
mcmc = MCMC(
    nuts_kernel, 
    num_warmup=100, 
    num_samples=100,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key, a0, P0)

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

In [ ]:
coefs_df

In [ ]:
state_labels = ["intercept"] + regressors
coefs_lookup = coefs_df.set_index("regressor")["coef"]

fig, axes = plt.subplots(len(state_labels), 1, figsize=(12, 2.5 * len(state_labels)), sharex=True)
for i, (ax, label) in enumerate(zip(axes, state_labels)):
    ax.plot(df["date"], at[:, i], linewidth=0.8)
    if i > 0 and label in coefs_lookup.index:
        ax.axhline(coefs_lookup[label], color="red", linestyle="--", linewidth=1.0, label="true coef")
        ax.legend(fontsize=8)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("date")
fig.suptitle("Filtered state coefficients (posterior median)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:

# mu_samples = np.array(at)
mu_samples = np.array(posteriors_dict["mu"])
sigma_samples = np.array(posteriors_dict["sigma_q"])

# generate error
eps_samples = np.transpose(
    np.random.normal(
        loc=0.0,
        scale=sigma_samples, 
        size=(len(y), 
        sigma_samples.shape[0])
    ),
    axes=(1, 0)
)
print(eps_samples.shape)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

In [ ]:
import matplotlib.pyplot as plt
n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')